# CBRAIN OASIS Baseline Modeling (Step 3A)

This notebook trains a deterministic scikit-learn logistic-regression baseline on frozen Step 2B preprocessing artifacts and reports leakage-safe validation metrics.

Environment note: run with a Python environment that has `scikit-learn` installed (project venv: `.venv`).

In [ ]:
# 1) Load frozen preprocessing artifacts
from __future__ import annotations

import csv
import json
from datetime import date
from pathlib import Path

import numpy as np

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('scikit-learn is required for Step 3A. Use the project venv: .venv/bin/python') from exc

cwd = Path.cwd().resolve()
candidate_roots = [cwd]
if cwd.name == 'notebooks':
    candidate_roots.append(cwd.parent)
else:
    candidate_roots.append(cwd / 'projects' / 'cbrain_oasis_cognition')
    candidate_roots.append(cwd.parent)

repo_root = None
for candidate in candidate_roots:
    if (candidate / 'results' / 'preprocessing' / 'X_train.csv').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate results/preprocessing/X_train.csv from current working directory.')

pre_dir = repo_root / 'results' / 'preprocessing'
cohort_dir = repo_root / 'results' / 'cohort'
model_dir = repo_root / 'results' / 'modeling'
docs_path = repo_root / 'docs' / 'modeling_step3a.md'

x_train_path = pre_dir / 'X_train.csv'
x_validation_path = pre_dir / 'X_validation.csv'
y_train_path = pre_dir / 'y_train.csv'
y_validation_path = pre_dir / 'y_validation.csv'
feature_manifest_path = pre_dir / 'feature_manifest.json'
split_subjects_path = cohort_dir / 'split_subjects.json'

feature_manifest = json.loads(feature_manifest_path.read_text(encoding='utf-8'))
split_payload = json.loads(split_subjects_path.read_text(encoding='utf-8'))

def _read_csv(path: Path) -> tuple[list[str], list[dict]]:
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        return list(reader.fieldnames or []), [dict(r) for r in reader]

x_train_cols, x_train_rows = _read_csv(x_train_path)
x_validation_cols, x_validation_rows = _read_csv(x_validation_path)
y_train_cols, y_train_rows = _read_csv(y_train_path)
y_validation_cols, y_validation_rows = _read_csv(y_validation_path)

print(f'Repo root: {repo_root}')
print(f'Train rows: {len(x_train_rows)}; Validation rows: {len(x_validation_rows)}')

In [ ]:
# 2) Contract checks and leakage gates
expected_feature_order = feature_manifest['final_feature_order']
if x_train_cols != expected_feature_order:
    raise AssertionError(f'X_train column order mismatch. Expected {expected_feature_order}, found {x_train_cols}')
if x_validation_cols != expected_feature_order:
    raise AssertionError(f'X_validation column order mismatch. Expected {expected_feature_order}, found {x_validation_cols}')

if y_train_cols != ['cognitive_impairment'] or y_validation_cols != ['cognitive_impairment']:
    raise AssertionError('Label files must contain only cognitive_impairment column.')

if len(x_train_rows) != len(y_train_rows):
    raise AssertionError('Train X/y row-count mismatch.')
if len(x_validation_rows) != len(y_validation_rows):
    raise AssertionError('Validation X/y row-count mismatch.')

excluded = {'CDR','Group','MMSE','Subject.ID','MRI.ID','rownames','Hand','Visit','MR.Delay'}
if any(col in excluded for col in x_train_cols):
    raise AssertionError('Leakage columns present in X_train.')
if any(col in excluded for col in x_validation_cols):
    raise AssertionError('Leakage columns present in X_validation.')

if split_payload['overlap_count'] != 0:
    raise AssertionError('Split leakage gate failed: overlap_count must be 0.')

train_subject_ids = sorted(split_payload['train_subject_ids'])
validation_subject_ids = sorted(split_payload['validation_subject_ids'])
if len(train_subject_ids) != len(x_train_rows) or len(validation_subject_ids) != len(x_validation_rows):
    raise AssertionError('Subject-id counts do not align with X row counts.')

def _validate_label_domain(rows: list[dict], split_name: str) -> None:
    vals = set()
    for r in rows:
        raw = str(r['cognitive_impairment']).strip()
        if raw == '':
            raise AssertionError(f'Missing label in {split_name}.')
        vals.add(int(raw))
    if not vals.issubset({0,1}):
        raise AssertionError(f'Invalid label domain in {split_name}: {vals}')

_validate_label_domain(y_train_rows, 'train')
_validate_label_domain(y_validation_rows, 'validation')

print('Leakage gates: PASS')

In [ ]:
# 3) Build matrices for deterministic training
def _rows_to_matrix(rows: list[dict], columns: list[str]) -> np.ndarray:
    return np.array([[float(r[c]) for c in columns] for r in rows], dtype=float)

def _rows_to_label(rows: list[dict]) -> np.ndarray:
    return np.array([int(r['cognitive_impairment']) for r in rows], dtype=int)

X_train = _rows_to_matrix(x_train_rows, expected_feature_order)
X_validation = _rows_to_matrix(x_validation_rows, expected_feature_order)
y_train = _rows_to_label(y_train_rows)
y_validation = _rows_to_label(y_validation_rows)

if np.isnan(X_train).any() or np.isnan(X_validation).any():
    raise AssertionError('NaN detected in feature matrix.')
print('X_train shape:', X_train.shape)
print('X_validation shape:', X_validation.shape)

In [ ]:
# 4) Train baseline logistic regression (scikit-learn, deterministic)
model = LogisticRegression(
    C=1.0,
    solver='liblinear',
    max_iter=2000,
    random_state=42,
)
model.fit(X_train, y_train)

train_prob = model.predict_proba(X_train)[:, 1]
validation_prob = model.predict_proba(X_validation)[:, 1]
train_pred = (train_prob >= 0.5).astype(int)
validation_pred = (validation_prob >= 0.5).astype(int)

train_info = {
    'C': 1.0,
    'solver': 'liblinear',
    'max_iter': 2000,
    'random_state': 42,
    'classes': model.classes_.astype(int).tolist(),
    'n_iter': model.n_iter_.astype(int).tolist(),
}
print('Model classes:', train_info['classes'])
print('n_iter:', train_info['n_iter'])

In [ ]:
# 5) Evaluate train and validation (ROC-AUC, PR-AUC, confusion matrix)
def _split_metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = [int(x) for x in cm.ravel()]
    return {
        'threshold': 0.5,
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'specificity': float(tn / (tn + fp) if (tn + fp) else 0.0),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'confusion_matrix': {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn},
    }

train_metrics = _split_metrics(y_train, train_prob, train_pred)
validation_metrics = _split_metrics(y_validation, validation_prob, validation_pred)

print('Validation ROC-AUC:', round(validation_metrics['roc_auc'], 4))
print('Validation PR-AUC :', round(validation_metrics['pr_auc'], 4))
print('Validation confusion:', validation_metrics['confusion_matrix'])

In [ ]:
# 6) Materialize Step 3A artifacts and documentation
model_dir.mkdir(parents=True, exist_ok=True)

coef_path = model_dir / 'baseline_logistic_coefficients.csv'
metrics_path = model_dir / 'baseline_metrics.json'
val_pred_path = model_dir / 'baseline_validation_predictions.csv'
train_pred_path = model_dir / 'baseline_train_predictions.csv'
model_card_path = model_dir / 'baseline_model_card.json'

coef_rows = [{'feature': 'intercept', 'coefficient': float(model.intercept_[0])}]
for idx, feature in enumerate(expected_feature_order):
    coef_rows.append({'feature': feature, 'coefficient': float(model.coef_[0][idx])})

with coef_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['feature', 'coefficient'])
    writer.writeheader()
    writer.writerows(coef_rows)

train_pred_rows = []
for sid, y_true, y_prob, y_hat in zip(train_subject_ids, y_train.tolist(), train_prob.tolist(), train_pred.tolist()):
    train_pred_rows.append({
        'Subject.ID': sid,
        'y_true': int(y_true),
        'y_probability': float(y_prob),
        'y_pred_threshold_0_5': int(y_hat),
    })

validation_pred_rows = []
for sid, y_true, y_prob, y_hat in zip(validation_subject_ids, y_validation.tolist(), validation_prob.tolist(), validation_pred.tolist()):
    validation_pred_rows.append({
        'Subject.ID': sid,
        'y_true': int(y_true),
        'y_probability': float(y_prob),
        'y_pred_threshold_0_5': int(y_hat),
    })

for path, rows in [(train_pred_path, train_pred_rows), (val_pred_path, validation_pred_rows)]:
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['Subject.ID','y_true','y_probability','y_pred_threshold_0_5'])
        writer.writeheader()
        writer.writerows(rows)

metrics_payload = {
    'model_name': 'baseline_logistic_sklearn',
    'date': date.today().isoformat(),
    'train_metrics': train_metrics,
    'validation_metrics': validation_metrics,
    'train_row_count': int(X_train.shape[0]),
    'validation_row_count': int(X_validation.shape[0]),
    'feature_order': expected_feature_order,
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

model_card = {
    'model_family': 'Logistic regression (scikit-learn)',
    'objective': 'Binary prediction of cognitive_impairment on frozen baseline cohort',
    'training_data': 'results/preprocessing/X_train.csv + y_train.csv',
    'validation_data': 'results/preprocessing/X_validation.csv + y_validation.csv',
    'fit_protocol': 'All preprocessing and split are frozen from Steps 2A/2B; no validation leakage in fitting.',
    'hyperparameters': train_info,
    'notes': [
        'Deterministic run with fixed random_state=42 and liblinear solver.',
        'Threshold is fixed at 0.5 for confusion metrics.',
        'No temporal or identifier features are used.',
    ],
}
model_card_path.write_text(json.dumps(model_card, indent=2, sort_keys=True) + '\n', encoding='utf-8')

run_date = date.today().isoformat()
docs_text = f"""# Baseline Modeling (Step 3A)

Date: {run_date}  
Notebook: `notebooks/04_model_baseline_step3a.ipynb`

## Scope

This step trains a deterministic **scikit-learn logistic regression** baseline on frozen Step 2 preprocessing outputs.

## Leakage Controls

- Inputs come only from `results/preprocessing/` and subject split from `results/cohort/split_subjects.json`.
- Excluded leakage fields (`CDR`, `Group`, `MMSE`, identifiers, `Visit`, `MR.Delay`) are absent from model matrix.
- Subject overlap check remains **0**.
- No refit of preprocessing on validation.

## Model

- Family: `sklearn.linear_model.LogisticRegression`
- Hyperparameters: C=1.0, solver=liblinear, max_iter=2000, random_state=42 (default L2).
- Feature order: {", ".join([f"`{x}`" for x in expected_feature_order])}.

## Metrics (Threshold = 0.5)

| Split | ROC-AUC | PR-AUC | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| Train | {train_metrics["roc_auc"]:.4f} | {train_metrics["pr_auc"]:.4f} | {train_metrics["accuracy"]:.4f} | {train_metrics["precision"]:.4f} | {train_metrics["recall"]:.4f} | {train_metrics["f1"]:.4f} |
| Validation | {validation_metrics["roc_auc"]:.4f} | {validation_metrics["pr_auc"]:.4f} | {validation_metrics["accuracy"]:.4f} | {validation_metrics["precision"]:.4f} | {validation_metrics["recall"]:.4f} | {validation_metrics["f1"]:.4f} |

Validation confusion matrix counts:
- TP={validation_metrics["confusion_matrix"]["tp"]}, TN={validation_metrics["confusion_matrix"]["tn"]}, FP={validation_metrics["confusion_matrix"]["fp"]}, FN={validation_metrics["confusion_matrix"]["fn"]}

## Artifacts

- `results/modeling/baseline_logistic_coefficients.csv`
- `results/modeling/baseline_metrics.json`
- `results/modeling/baseline_train_predictions.csv`
- `results/modeling/baseline_validation_predictions.csv`
- `results/modeling/baseline_model_card.json`
"""
docs_path.write_text(docs_text, encoding='utf-8')

print('Wrote:', coef_path)
print('Wrote:', metrics_path)
print('Wrote:', train_pred_path)
print('Wrote:', val_pred_path)
print('Wrote:', model_card_path)
print('Wrote:', docs_path)